In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("ingestion_date", "")
dbutils.widgets.text("run_id", "")

INGESTION_DATE = dbutils.widgets.get("ingestion_date").strip()
RUN_ID = dbutils.widgets.get("run_id").strip()

SILVER_PATH = "s3://johnq-data-lake-dev/abs-building-approvals/silver/"
GOLD_PATH   = "s3://johnq-data-lake-dev/abs-building-approvals/gold/"

df = (spark.read.format("delta").load(SILVER_PATH)
      .filter(F.col("ingestion_date") == INGESTION_DATE))

df.createOrReplaceTempView("silver_filtered")


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW gold_abs_building_approvals AS
SELECT
  time_period,
  region_code,
  region_label,
  building_type_label,
  measure_label,
  sector_label,
  work_type_label,
  SUM(obs_value) AS obs_value_total
FROM silver_filtered
GROUP BY
  time_period,
  region_code,
  region_label,
  building_type_label,
  measure_label,
  sector_label,
  work_type_label;


In [0]:
gold = spark.table("gold_abs_building_approvals").withColumn("ingestion_date", F.lit(INGESTION_DATE))

(gold.write.format("delta")
  .mode("overwrite")
  .option("replaceWhere", f"ingestion_date = '{INGESTION_DATE}'")
  .partitionBy("ingestion_date")
  .save(GOLD_PATH)
)


In [0]:
spark.read.format("delta").load(GOLD_PATH).limit(5).show(truncate = False)